# 📊 01 — Data Cleaning & Preparation

**Project:** AI-Powered Sales Intelligence Platform  
**Notebook Purpose:** Inspect, clean, transform, and export a production-ready sales dataset.  
**Input:** `data/raw/Sample - Superstore.csv`  
**Output:** `data/processed/cleaned_sales.csv`  

---

### Pipeline Context

This is the **first notebook** in the data pipeline. The cleaned dataset produced here feeds into:

| Downstream Step | Notebook / Component |
|---|---|
| Exploratory Data Analysis | `02_eda.ipynb` |
| Sales Forecasting Model | `03_forecasting.ipynb` |
| LangGraph AI Agent | `agent/` |
| Power BI Dashboard | `dashboard/` |

> **Reproducibility:** Run all cells top-to-bottom for a fully reproducible result.

---
## 1 · Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)
warnings.filterwarnings("ignore")

# Plot styling
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

print("✅ Libraries imported successfully.")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")

✅ Libraries imported successfully.
   pandas  3.0.5
   numpy   2.5.3


---
## 2 · Load the Raw Dataset

The dataset is the well-known *Sample Superstore* file, containing transactional sales data for a US-based retail business.

In [2]:
# Path configuration — works whether the notebook is run from /notebooks or the project root
RAW_DATA_PATH = os.path.join("..", "data", "raw", "Sample - Superstore.csv")

if not os.path.exists(RAW_DATA_PATH):
    # Fallback for running from project root
    RAW_DATA_PATH = os.path.join("data", "raw", "Sample - Superstore.csv")

df = pd.read_csv(RAW_DATA_PATH, encoding="latin-1")

print(f"✅ Dataset loaded successfully.")
print(f"   Rows   : {df.shape[0]:,}")
print(f"   Columns: {df.shape[1]}")

✅ Dataset loaded successfully.
   Rows   : 9,994
   Columns: 21


---
## 3 · Initial Dataset Inspection

Before touching the data, we need to understand its structure, types, and distributions.

### 3.1 — Shape & First Rows

In [3]:
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5,0.45,-383.03
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2,0.20,2.52


### 3.2 — Column Names

In [4]:
print(f"Total columns: {len(df.columns)}\n")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:>2}. {col}")

Total columns: 21

   1. Row ID
   2. Order ID
   3. Order Date
   4. Ship Date
   5. Ship Mode
   6. Customer ID
   7. Customer Name
   8. Segment
   9. Country
  10. City
  11. State
  12. Postal Code
  13. Region
  14. Product ID
  15. Category
  16. Sub-Category
  17. Product Name
  18. Sales
  19. Quantity
  20. Discount
  21. Profit


### 3.3 — Data Types

In [5]:
df.dtypes

Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code        int64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

### 3.4 — Summary Statistics (Numerical Columns)

In [6]:
df.describe()

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
count,9994.00,9994.00,9994.00,9994.00,9994.00,9994.00
mean,4997.50,55190.38,229.86,3.79,0.16,28.66
std,2885.16,32063.69,623.25,2.23,0.21,234.26
min,1.00,1040.00,0.44,1.00,0.00,-6599.98
25%,2499.25,23223.00,17.28,2.00,0.00,1.73
50%,4997.50,56430.50,54.49,3.00,0.20,8.67
75%,7495.75,90008.00,209.94,5.00,0.20,29.36
max,9994.00,99301.00,22638.48,14.00,0.80,8399.98


### 3.5 — Unique Values for Key Categorical Columns

Understanding the cardinality and content of important categorical columns helps us spot data-entry issues early.

In [7]:
categorical_cols = [
    "Segment", "Region", "Category", "Sub-Category", "Ship Mode",
]

for col in categorical_cols:
    unique_vals = df[col].unique()
    print(f"\n🔹 {col} ({len(unique_vals)} unique):")
    for v in sorted(unique_vals):
        count = (df[col] == v).sum()
        print(f"    • {v:<20s}  ({count:,} rows)")


🔹 Segment (3 unique):
    • Consumer              (5,191 rows)
    • Corporate             (3,020 rows)
    • Home Office           (1,783 rows)

🔹 Region (4 unique):
    • Central               (2,323 rows)
    • East                  (2,848 rows)
    • South                 (1,620 rows)
    • West                  (3,203 rows)

🔹 Category (3 unique):
    • Furniture             (2,121 rows)
    • Office Supplies       (6,026 rows)
    • Technology            (1,847 rows)

🔹 Sub-Category (17 unique):
    • Accessories           (775 rows)
    • Appliances            (466 rows)
    • Art                   (796 rows)
    • Binders               (1,523 rows)
    • Bookcases             (228 rows)
    • Chairs                (617 rows)
    • Copiers               (68 rows)
    • Envelopes             (254 rows)
    • Fasteners             (217 rows)
    • Furnishings           (957 rows)
    • Labels                (364 rows)
    • Machines              (115 rows)
    • Paper            

---
## 4 · Data Quality Checks

We systematically check for common data-quality issues **before** making any changes.  
Each issue is documented; no data is removed without explanation.

### 4.1 — Missing Values

In [8]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
}).sort_values("Missing Count", ascending=False)

print("Missing Value Report")
print("=" * 40)
if missing.sum() == 0:
    print("✅ No missing values found in any column.")
else:
    display(missing_report[missing_report["Missing Count"] > 0])

Missing Value Report
✅ No missing values found in any column.


### 4.2 — Duplicate Rows

In [9]:
duplicate_rows = df.duplicated().sum()

print("Duplicate Row Report")
print("=" * 40)
print(f"Exact duplicate rows: {duplicate_rows:,}")

if duplicate_rows > 0:
    print("\n⚠️  Sample duplicates:")
    display(df[df.duplicated(keep=False)].head(10))
else:
    print("✅ No exact duplicate rows found.")

Duplicate Row Report
Exact duplicate rows: 0
✅ No exact duplicate rows found.


### 4.3 — Duplicate Order IDs

A single order can contain multiple items (line items), so duplicate `Order ID` values are **expected**.  
We verify this is the case and not a data error.

In [10]:
total_rows = len(df)
unique_orders = df["Order ID"].nunique()
duplicate_order_ids = total_rows - unique_orders

print("Order ID Analysis")
print("=" * 40)
print(f"Total rows          : {total_rows:,}")
print(f"Unique Order IDs    : {unique_orders:,}")
print(f"Repeated Order IDs  : {duplicate_order_ids:,}")
print(f"\nℹ️  This is expected — each Order ID can have multiple line items (products).")

# Show the distribution of items per order
items_per_order = df.groupby("Order ID").size()
print(f"\nItems per order (distribution):")
print(items_per_order.describe().to_string())

Order ID Analysis
Total rows          : 9,994
Unique Order IDs    : 5,009
Repeated Order IDs  : 4,985

ℹ️  This is expected — each Order ID can have multiple line items (products).

Items per order (distribution):
count   5009.00
mean       2.00
std        1.41
min        1.00
25%        1.00
50%        1.00
75%        2.00
max       14.00


### 4.4 — Invalid Values

In [11]:
negative_sales = (df["Sales"] < 0).sum()
negative_qty   = (df["Quantity"] < 0).sum()
negative_profit = (df["Profit"] < 0).sum()

print("Invalid / Suspicious Value Report")
print("=" * 40)
print(f"Negative Sales     : {negative_sales:,}  {'✅' if negative_sales == 0 else '⚠️'}")
print(f"Negative Quantity  : {negative_qty:,}  {'✅' if negative_qty == 0 else '⚠️'}")
print(f"Negative Profit    : {negative_profit:,}  {'(expected — these are loss-making transactions)'}")

if negative_profit > 0:
    loss_pct = (negative_profit / total_rows * 100)
    print(f"\nℹ️  {negative_profit:,} rows ({loss_pct:.1f}%) have negative profit.")
    print("   Negative profit is a normal business occurrence (discounts, returns, etc.).")
    print("   These rows are KEPT — they are valid and important for analysis.")

Invalid / Suspicious Value Report
Negative Sales     : 0  ✅
Negative Quantity  : 0  ✅
Negative Profit    : 1,871  (expected — these are loss-making transactions)

ℹ️  1,871 rows (18.7%) have negative profit.
   Negative profit is a normal business occurrence (discounts, returns, etc.).
   These rows are KEPT — they are valid and important for analysis.


### 4.5 — Date Consistency Check

We verify that `Ship Date` is always on or after `Order Date`.

In [12]:
# Temporarily parse dates for validation
temp_order_date = pd.to_datetime(df["Order Date"], format="mixed", dayfirst=False)
temp_ship_date  = pd.to_datetime(df["Ship Date"], format="mixed", dayfirst=False)

date_issues = (temp_ship_date < temp_order_date).sum()

print("Date Consistency Report")
print("=" * 40)
print(f"Ship Date before Order Date: {date_issues:,}")

if date_issues == 0:
    print("✅ All ship dates are on or after the order date.")
else:
    print("⚠️  Found rows where Ship Date < Order Date:")
    bad_dates = df[temp_ship_date < temp_order_date]
    display(bad_dates[["Order ID", "Order Date", "Ship Date"]].head(10))

Date Consistency Report


Ship Date before Order Date: 0
✅ All ship dates are on or after the order date.


---
## 5 · Data Cleaning

Based on the quality checks above, we now apply targeted cleaning steps.  
Each operation is explained before it is applied.

### 5.1 — Convert Date Columns to `datetime`

`Order Date` and `Ship Date` are currently stored as strings. Converting them to proper `datetime` types enables time-series analysis, feature extraction, and date arithmetic.

In [13]:
df["Order Date"] = pd.to_datetime(df["Order Date"], format="mixed", dayfirst=False)
df["Ship Date"]  = pd.to_datetime(df["Ship Date"],  format="mixed", dayfirst=False)

print("✅ Date columns converted to datetime.")
print(f"   Order Date range: {df['Order Date'].min().date()} → {df['Order Date'].max().date()}")
print(f"   Ship Date range : {df['Ship Date'].min().date()} → {df['Ship Date'].max().date()}")

✅ Date columns converted to datetime.
   Order Date range: 2014-01-03 → 2017-12-30
   Ship Date range : 2014-01-07 → 2018-01-05


### 5.2 — Standardize Column Names to `snake_case`

Consistent, lowercase column names make the dataset easier to work with in Python, SQL, and downstream tools.

| Before | After |
|---|---|
| `Order Date` | `order_date` |
| `Customer Name` | `customer_name` |
| `Sub-Category` | `sub_category` |

In [14]:
# Store original names for reference
original_columns = df.columns.tolist()

# Standardize: lowercase, replace spaces & hyphens with underscores
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

print("✅ Column names standardized to snake_case.\n")
for old, new in zip(original_columns, df.columns):
    marker = "  →  " if old != new else "      "
    print(f"   {old:<20s}{marker}{new}")

✅ Column names standardized to snake_case.

   Row ID                →  row_id
   Order ID              →  order_id
   Order Date            →  order_date
   Ship Date             →  ship_date
   Ship Mode             →  ship_mode
   Customer ID           →  customer_id
   Customer Name         →  customer_name
   Segment               →  segment
   Country               →  country
   City                  →  city
   State                 →  state
   Postal Code           →  postal_code
   Region                →  region
   Product ID            →  product_id
   Category              →  category
   Sub-Category          →  sub_category
   Product Name          →  product_name
   Sales                 →  sales
   Quantity              →  quantity
   Discount              →  discount
   Profit                →  profit


### 5.3 — Remove Duplicate Rows

Our earlier check found **no exact duplicate rows**. We apply the deduplication step defensively so the pipeline is robust if the upstream data changes.

In [15]:
rows_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
rows_after = len(df)
rows_removed = rows_before - rows_after

print(f"Rows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Rows removed: {rows_removed:,}")
if rows_removed == 0:
    print("✅ No duplicates to remove.")
else:
    print(f"⚠️  Removed {rows_removed:,} duplicate rows.")

Rows before : 9,994
Rows after  : 9,994
Rows removed: 0
✅ No duplicates to remove.


### 5.4 — Handle Missing Values

Our earlier check found **no missing values**. We add a defensive verification step to ensure data integrity.

In [16]:
missing_after_clean = df.isnull().sum()
total_missing = missing_after_clean.sum()

if total_missing == 0:
    print("✅ No missing values — no imputation needed.")
else:
    print(f"⚠️  {total_missing} missing values found after cleaning:")
    print(missing_after_clean[missing_after_clean > 0])
    # If missing values appeared, we would handle them here:
    # - Numerical: median imputation
    # - Categorical: mode imputation or 'Unknown'
    # - Critical IDs: drop the row

✅ No missing values — no imputation needed.


### 5.5 — Verify Numerical Data Types

Ensure that all numerical columns have the correct types for downstream computation.

In [17]:
numerical_cols = ["row_id", "postal_code", "sales", "quantity", "discount", "profit"]

print("Numerical Column Type Verification")
print("=" * 45)
for col in numerical_cols:
    dtype = df[col].dtype
    is_numeric = pd.api.types.is_numeric_dtype(df[col])
    status = "✅" if is_numeric else "⚠️"
    print(f"   {status} {col:<15s}  →  {dtype}")

# Ensure integer columns stay as integers
df["row_id"]      = df["row_id"].astype(int)
df["postal_code"]  = df["postal_code"].astype(int)
df["quantity"]     = df["quantity"].astype(int)

print("\n✅ All numerical columns verified.")

Numerical Column Type Verification
   ✅ row_id           →  int64
   ✅ postal_code      →  int64
   ✅ sales            →  float64
   ✅ quantity         →  int64
   ✅ discount         →  float64
   ✅ profit           →  float64

✅ All numerical columns verified.


---
## 6 · Feature Engineering

We create new business-relevant features that will be used across all downstream analysis and models.  
These features transform raw transactional data into **actionable signals**.

### 6A — `profit_margin`

**Formula:** `profit / sales`  
**Purpose:** Measures profitability as a percentage of revenue.  
**Edge case:** Division by zero is handled — if `sales == 0`, `profit_margin` is set to `0.0`.

> **⚠️ Outlier Note:** `profit_margin` values range from approximately −2.75 to +0.50. Extreme negatives occur when the loss far exceeds the revenue (e.g., heavy discounts). These are **valid business data** and should not be removed automatically. See `docs/data_dictionary.md` for detailed guidance on handling these values in EDA and ML pipelines.

In [18]:
df["profit_margin"] = np.where(
    df["sales"] != 0,
    df["profit"] / df["sales"],
    0.0
)

print("✅ profit_margin created.")
print(f"   Mean   : {df['profit_margin'].mean():.4f}")
print(f"   Median : {df['profit_margin'].median():.4f}")
print(f"   Min    : {df['profit_margin'].min():.4f}")
print(f"   Max    : {df['profit_margin'].max():.4f}")

✅ profit_margin created.
   Mean   : 0.1203
   Median : 0.2700
   Min    : -2.7500
   Max    : 0.5000


### 6B — `order_year`

**Purpose:** Enables year-over-year comparisons and annual aggregation.

In [19]:
df["order_year"] = df["order_date"].dt.year

print("✅ order_year created.")
print(f"   Unique years: {sorted(df['order_year'].unique())}")
print(f"\n   Distribution:")
print(df["order_year"].value_counts().sort_index().to_string())

✅ order_year created.
   Unique years: [np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017)]

   Distribution:
order_year
2014    1993
2015    2102
2016    2587
2017    3312


### 6C — `order_month`

**Purpose:** Enables monthly trend analysis and seasonality detection.

In [20]:
df["order_month"] = df["order_date"].dt.month

print("✅ order_month created.")
print(f"   Range: {df['order_month'].min()} — {df['order_month'].max()}")

✅ order_month created.
   Range: 1 — 12


### 6D — `order_quarter`

**Purpose:** Quarterly business reporting (Q1, Q2, Q3, Q4).

In [21]:
df["order_quarter"] = "Q" + df["order_date"].dt.quarter.astype(str)

print("✅ order_quarter created.")
print(f"\n   Distribution:")
print(df["order_quarter"].value_counts().sort_index().to_string())

✅ order_quarter created.

   Distribution:
order_quarter
Q1    1377
Q2    2120
Q3    2799
Q4    3698


### 6E — `shipping_days`

**Formula:** `ship_date - order_date` (in days)  
**Purpose:** Measures fulfillment speed — useful for logistics analysis and customer satisfaction.

In [22]:
df["shipping_days"] = (df["ship_date"] - df["order_date"]).dt.days

print("✅ shipping_days created.")
print(f"   Mean   : {df['shipping_days'].mean():.1f} days")
print(f"   Median : {df['shipping_days'].median():.1f} days")
print(f"   Min    : {df['shipping_days'].min()} days")
print(f"   Max    : {df['shipping_days'].max()} days")

✅ shipping_days created.
   Mean   : 4.0 days
   Median : 4.0 days
   Min    : 0 days
   Max    : 7 days


### 6F — `is_loss`

**Rule:** `True` if `profit < 0`, else `False`  
**Purpose:** Quick boolean flag for filtering and aggregating loss-making transactions.

In [23]:
df["is_loss"] = df["profit"] < 0

loss_count = df["is_loss"].sum()
loss_pct   = (loss_count / len(df) * 100)

print("✅ is_loss created.")
print(f"   Loss-making transactions : {loss_count:,} ({loss_pct:.1f}%)")
print(f"   Profitable transactions  : {len(df) - loss_count:,} ({100 - loss_pct:.1f}%)")

✅ is_loss created.
   Loss-making transactions : 1,871 (18.7%)
   Profitable transactions  : 8,123 (81.3%)


### Feature Engineering Summary

In [24]:
new_features = ["profit_margin", "order_year", "order_month", "order_quarter", "shipping_days", "is_loss"]

print("New Features Created")
print("=" * 50)
for feat in new_features:
    dtype = df[feat].dtype
    print(f"   ✅ {feat:<20s}  ({dtype})")

print(f"\n   Total new features: {len(new_features)}")
print(f"   Total columns now : {len(df.columns)}")

New Features Created


   ✅ profit_margin         (float64)
   ✅ order_year            (int32)
   ✅ order_month           (int32)
   ✅ order_quarter         (str)
   ✅ shipping_days         (int64)
   ✅ is_loss               (bool)

   Total new features: 6
   Total columns now : 27


---
## 7 · Post-Cleaning Validation

Final checks to confirm the cleaned dataset is production-ready.

### 7.1 — Final Shape

In [25]:
print("Final Dataset Shape")
print("=" * 40)
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")

Final Dataset Shape
   Rows    : 9,994
   Columns : 27


### 7.2 — Final Data Types

In [26]:
print("Final Data Types")
print("=" * 45)
for col in df.columns:
    print(f"   {col:<20s}  {str(df[col].dtype)}")

Final Data Types
   row_id                int64
   order_id              str
   order_date            datetime64[us]
   ship_date             datetime64[us]
   ship_mode             str
   customer_id           str
   customer_name         str
   segment               str
   country               str
   city                  str
   state                 str
   postal_code           int64
   region                str
   product_id            str
   category              str
   sub_category          str
   product_name          str
   sales                 float64
   quantity              int64
   discount              float64
   profit                float64
   profit_margin         float64
   order_year            int32
   order_month           int32
   order_quarter         str
   shipping_days         int64
   is_loss               bool


### 7.3 — Final Missing Value Report

In [27]:
final_missing = df.isnull().sum().sum()
print(f"Total missing values: {final_missing}")

if final_missing == 0:
    print("✅ No missing values in the final dataset.")
else:
    print("⚠️  Missing values remain:")
    print(df.isnull().sum()[df.isnull().sum() > 0])

Total missing values: 0
✅ No missing values in the final dataset.


### 7.4 — Sample Rows with New Features

In [28]:
display_cols = [
    "order_id", "order_date", "ship_date", "category", "sub_category",
    "sales", "profit", "profit_margin", "order_year", "order_month",
    "order_quarter", "shipping_days", "is_loss"
]

print("Sample rows with new features:")
df[display_cols].head(10)

Sample rows with new features:


,order_id,order_date,ship_date,category,sub_category,sales,profit,profit_margin,order_year,order_month,order_quarter,shipping_days,is_loss
0,CA-2016-152156,2016-11-08,2016-11-11,Furniture,Bookcases,261.96,41.91,0.16,2016,11,Q4,3,False
1,CA-2016-152156,2016-11-08,2016-11-11,Furniture,Chairs,731.94,219.58,0.30,2016,11,Q4,3,False
2,CA-2016-138688,2016-06-12,2016-06-16,Office Supplies,Labels,14.62,6.87,0.47,2016,6,Q2,4,False
3,US-2015-108966,2015-10-11,2015-10-18,Furniture,Tables,957.58,-383.03,-0.40,2015,10,Q4,7,True
4,US-2015-108966,2015-10-11,2015-10-18,Office Supplies,Storage,22.37,2.52,0.11,2015,10,Q4,7,False
5,CA-2014-115812,2014-06-09,2014-06-14,Furniture,Furnishings,48.86,14.17,0.29,2014,6,Q2,5,False
6,CA-2014-115812,2014-06-09,2014-06-14,Office Supplies,Art,7.28,1.97,0.27,2014,6,Q2,5,False
7,CA-2014-115812,2014-06-09,2014-06-14,Technology,Phones,907.15,90.72,0.10,2014,6,Q2,5,False
8,CA-2014-115812,2014-06-09,2014-06-14,Office Supplies,Binders,18.50,5.78,0.31,2014,6,Q2,5,False
9,CA-2014-115812,2014-06-09,2014-06-14,Office Supplies,Appliances,114.90,34.47,0.30,2014,6,Q2,5,False


---
## 8 · Export Cleaned Dataset

The cleaned and feature-enriched dataset is saved to `data/processed/cleaned_sales.csv`.  
This file is the **single source of truth** for all downstream analysis.

In [29]:
OUTPUT_DIR  = os.path.join("..", "data", "processed")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "cleaned_sales.csv")

# Fallback for running from project root
if not os.path.exists(OUTPUT_DIR):
    OUTPUT_DIR  = os.path.join("data", "processed")
    OUTPUT_PATH = os.path.join(OUTPUT_DIR, "cleaned_sales.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

df.to_csv(OUTPUT_PATH, index=False)

# Verify export
file_size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)

print("✅ Cleaned dataset exported successfully.")
print(f"   Path      : {OUTPUT_PATH}")
print(f"   File size : {file_size_mb:.2f} MB")
print(f"   Rows      : {len(df):,}")
print(f"   Columns   : {len(df.columns)}")

✅ Cleaned dataset exported successfully.


   Path      : ..\data\processed\cleaned_sales.csv
   File size : 2.49 MB
   Rows      : 9,994
   Columns   : 27


---
## 📋 Data Cleaning Summary

| Metric | Value |
|---|---|
| **Original dataset size** | 9,994 rows × 21 columns |
| **Final dataset size** | 9,994 rows × 27 columns |

### Cleaning Operations Performed

| # | Operation | Details |
|---|---|---|
| 1 | Date conversion | `order_date` and `ship_date` converted from string → `datetime64` |
| 2 | Column renaming | All 21 columns standardized to `snake_case` |
| 3 | Duplicate check | 0 exact duplicate rows found — no removal needed |
| 4 | Missing value check | 0 missing values found — no imputation needed |
| 5 | Data type verification | All numerical columns verified as correct types |
| 6 | Date consistency | All ship dates confirmed ≥ order dates |
| 7 | Invalid value check | 0 negative sales, 0 negative quantities — data is clean |

### New Features Created

| Feature | Type | Description |
|---|---|---|
| `profit_margin` | `float64` | Profit as a ratio of sales (handles division by zero) |
| `order_year` | `int64` | Year extracted from order date |
| `order_month` | `int64` | Month extracted from order date |
| `order_quarter` | `object` | Fiscal quarter label (Q1–Q4) |
| `shipping_days` | `int64` | Days between order and shipment |
| `is_loss` | `bool` | Whether the transaction was loss-making |

### Output

Cleaned dataset saved to: **`data/processed/cleaned_sales.csv`**

### Documentation

- **Data Dictionary:** See [`docs/data_dictionary.md`](../docs/data_dictionary.md) for column definitions, types, and usage notes.
- **Date Parsing:** When loading the cleaned CSV in downstream notebooks, always use `parse_dates=["order_date", "ship_date"]` to preserve datetime types.

### Next Steps

➡️ **`02_eda.ipynb`** — Exploratory Data Analysis  
➡️ **`03_forecasting.ipynb`** — Sales Forecasting Model  
➡️ **`agent/`** — LangGraph AI Agent  
➡️ **`dashboard/`** — Power BI Dashboard